In [ ]:
import os
import re
import sys
from functools import reduce

import pandas as pd

# Every path below is absolute, resolved from the repo root, so this notebook behaves the
# same in Jupyter (cwd = this directory) as under `./run.py json-transformer`.
repo = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(repo, 'src', 'assets', 'data')):
    repo, prev = os.path.dirname(repo), repo
    assert repo != prev, 'run this from inside the wingsearch checkout'

sheets_dir = os.path.join(repo, 'scripts', 'transform')
rulings_dir = os.path.join(repo, 'scripts', 'rulings')
data_dir = os.path.join(repo, 'src', 'assets', 'data')

sys.path.insert(0, rulings_dir)
import general_map

pd.set_option('display.max_rows', 500)

In [ ]:
cards = os.path.join(sheets_dir, 'wingspan-card-list.xlsx')
notes = os.path.join(sheets_dir, 'wingspan-note-list.xlsx')

master = pd.read_excel(cards, sheet_name='Birds')
hummingbirds = pd.read_excel(cards, sheet_name='Hummingbirds')
bonus = pd.read_excel(notes, sheet_name='Bonus')
goals = pd.read_excel(cards, sheet_name='Goals')
note = pd.read_excel(notes, sheet_name='Birds')
parameters = pd.read_excel(notes, sheet_name='Parameters', index_col=0)

In [3]:
expansion_order = {
    'originalcore': 0,
    'swiftstart': 0,
    'core': 0,
    'european': 0,
    'oceania': 1,
    'asia': 2,
    'promoAsia': 3,
    'promoCA': 4,
    'promoEurope': 5,
    'promoNZ': 6,
    'promoUK': 7,
    'promoUS': 8,
    'americas': 9
}

def sort_key(x):
    if x.name == 'Set':
        return x.map(expansion_order)
    return x

master.dropna(subset=['Common name'], inplace=True)
master.sort_values(by=['Set', 'Common name'], inplace=True, ignore_index=True, key=sort_key)
master['id'] = master.index + 2
master['Common name'] = master['Common name'].map(lambda s: s.strip())
master['Native name'] = note['Native name']
master['Note'] = note['Note']
master.loc[pd.isna(master['Nest type']), 'Nest type'] = 'none'
master['CardType'] = 'Bird'

hummingbirds.sort_values(by=['Group', 'Common name'], inplace=True, ignore_index=True, key=sort_key)
hummingbirds['Set'] = 'americas'
hummingbirds['Color'] = 'white'
hummingbirds['Victory points'] = 0
hummingbirds['Nest type'] = 'none'
hummingbirds['Egg limit'] = 0
hummingbirds['Wingspan'] = 0
hummingbirds['Forest'] = 'X'
hummingbirds['Grassland'] = 'X'
hummingbirds['Wetland'] = 'X'
hummingbirds['Total food cost'] = 0
hummingbirds['Backyard Birder'] = 'X'
hummingbirds['Bird Bander'] = 'X'
hummingbirds['Passerine Specialist'] = 'X'
hummingbirds['Small Clutch Specialist'] = 'X'
hummingbirds['id'] = hummingbirds.index + 2000
hummingbirds['CardType'] = 'Hummingbird'

bonus.sort_values(by=['Set', 'Bonus card'], inplace=True, ignore_index=True, key=sort_key)
bonus['id'] = bonus.index + 1000
bonus['CardType'] = 'Bonus'

goals.sort_values(by=['Set', 'Goal'], inplace=True, ignore_index=True, key=sort_key)
goals['id'] = goals.index + 2000

/var/folders/8h/6vlx40wx30nfgp121_wfbqn00000gq/T/ipykernel_68527/1550445390.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  master['Nest type'].loc[pd.isna(master['Nest type'])] = 'none'


In [ ]:
master_ids = reduce(lambda acc, val: {**acc, val[1]['Common name']: val[1]['id']}, master.iterrows(), {})
hummingbirds_ids = reduce(lambda acc, val: {**acc, val[1]['Common name']: val[1]['id']}, hummingbirds.iterrows(), {})
bonus_ids = reduce(lambda acc, val: {**acc, val[1]['Bonus card']: val[1]['id']}, bonus.iterrows(), {})
ids = {**master_ids, **bonus_ids, **hummingbirds_ids}

def transform_links(link: str):
    name = re.findall(r' applink="/card/([^"]+)"', link)[0]
    return f' applink="/card/{ids[name]}"' if name in ids else ''

# The TSV is written in a LaTeX-ish shorthand; anything not translated here reaches the
# player as literal text. `---`/`--` are LaTeX's em and en dash, and the corpus uses both
# (as a parenthetical dash and in ranges like 4--5), so translate them too -- longest
# first, or `---` would be eaten as `--` plus a stray hyphen.
markup = [
    (r"''", r'"</i>'),
    (r'``', r'<i>"'),
    (r'\\textit\{([^}]+)\}', r'<i>\1</i>'),
    (r'\\textbf\{([^}]+)\}', r'<strong applink="/card/\1">\1</strong>'),
    (r'---', '\u2014'),
    (r'--', '\u2013'),
]

def transform_markup(text: str):
    return reduce(lambda acc, sub: re.sub(sub[0], sub[1], acc), markup, text)

rulings = pd.read_csv(os.path.join(rulings_dir, 'rulings.tsv'), sep='\t', header=None, names=['id', 'general', 'specific', 'text', 'source'])
rulings['text'] = rulings.text.map(transform_markup)
rulings['text'] = rulings.text.map(lambda text: reduce(lambda acc, val: acc.replace(val, transform_links(val)), [x.group() for x in re.finditer( r' applink="/card/([^"]+)"', text)], text))

general = rulings[~pd.isna(rulings['general'])].drop(['specific'], axis=1)
general['general'] = general.general.map(lambda t: re.sub(r'\$\\Rightarrow\$', '➔', t))
specific = rulings[~pd.isna(rulings['specific'])].drop(['general'], axis=1)
specific['specific'] = specific['specific'].map(lambda s: s.strip())
grouped = specific.groupby(by='specific').apply(lambda group: list(map(lambda t: {'text': t[0], 'source': t[1]}, zip(group['text'], group['source']))))

In [ ]:
# A ruling whose card name matches nothing is dropped in silence: `grouped[name]` below is
# only consulted for names that exist, so a typo costs a player a ruling and nothing says so.
# "Greater Prairie Chicken" sat here unnoticed (the card is "Greater Prairie-Chicken"), which
# is why this is an assertion and not a printout.
vals =  list(master['Common name'].values) + list(bonus['Bonus card'].values) + list(hummingbirds['Common name'].values)
unmatched = list(filter(lambda g: g not in vals, grouped.index))
assert not unmatched, (
    f'rulings.tsv names cards that do not exist: {unmatched}. Fix the spelling in the TSV '
    '(these rulings are currently reaching no card).')

In [ ]:
general_dict = reduce(lambda acc, val: {**acc, val: []}, list(master['Common name']) + list(bonus['Bonus card']) + list(hummingbirds['Common name']), {})
rule_counts = {}

for i, rule in general.iterrows():
    # A general row with no predicate reaches no card, so this must fail rather than print.
    # Add an entry to general_map.candidates -- `lambda row: False` if it genuinely belongs
    # on no bird, with a comment saying why.
    assert rule['id'] in general_map.rulings, (
        f'general ruling {rule["id"]} ({rule["general"]}) has no predicate in general_map.candidates')
    rule_counter = 0
    for j, row in master.iterrows():
        if general_map.rulings[rule['id']](row):
            rule_counter += 1
            general_dict[master.loc[j, 'Common name']] += [{'id': rule['id'], 'text': rule['text'], 'source': rule['source']}]
    rule_counts[rule['id']] = rule_counter

for rules in general_dict.values():
    rules.sort(key=lambda key: rule_counts[key['id']])
    for rule in rules:
        del rule['id']

rule_counts

In [7]:
general.reset_index(drop=True, inplace=True)
general.drop(['id'], axis=1, inplace=True)
general.columns = ['name', 'text', 'source']
master['rulings'] = master['Common name'].map(lambda name: grouped[name] if name in grouped else [])
master['additionalRulings'] = master['Common name'].map(lambda name: general_dict[name])
bonus['rulings'] = bonus['Bonus card'].map(lambda name: grouped[name] if name in grouped else [])
bonus['%'] = bonus['%'].map(lambda p: int(p) if type(p) == float else p)

In [8]:
master.sort_values(by='Common name', inplace=True)
bonus.sort_values(by='Bonus card', inplace=True)
hummingbirds.sort_values(by='Common name', inplace=True)

In [ ]:
master.to_json(os.path.join(data_dir, 'master.json'), orient='records', indent=2)
hummingbirds.to_json(os.path.join(data_dir, 'hummingbirds.json'), orient='records', indent=2)
bonus.to_json(os.path.join(data_dir, 'bonus.json'), orient='records', indent=2)
general.to_json(os.path.join(data_dir, 'general.json'), orient='index', indent=2)
goals.to_json(os.path.join(data_dir, 'goals.json'), orient='records', indent=2)
parameters.to_json(os.path.join(data_dir, 'parameters.json'), orient='index', indent=2)